# 🌅 Uma Tardezinha na Beira do Guaíba
## Estatística e Análise de Dados na prática

Bem-vindo(a)! Este notebook conta uma história — um passeio de sábado na orla do Guaíba — e usa essa história pra ensinar, passo a passo, os principais conceitos de **estatística** e **análise de dados**.

Você vai poder **rodar** cada trecho de código, mudar valores, testar suas próprias ideias e ver os resultados na hora — é assim que se aprende estatística de verdade: brincando com números reais.

📖 Este notebook acompanha o arquivo `material_teorico.md`, que tem a mesma história com explicações mais completas. Recomenda-se ler os dois lado a lado.

**Como usar no Google Colab:**
1. Rode a primeira célula de código (importações) antes de qualquer outra.
2. Vá rodando as células em ordem, de cima pra baixo (Shift + Enter).
3. Sinta-se livre pra mexer nos números e ver o que muda!


In [ ]:
# 📦 Bibliotecas que vamos usar ao longo do notebook
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import random

# Deixa os gráficos com um tamanho legível
plt.rcParams['figure.figsize'] = (8, 5)
plt.rcParams['font.size'] = 11

print("Tudo pronto! Bora pra orla do Guaíba. 🌊")


***
## Capítulo 1 — O convite

É sábado de manhã. O grupo da escola combinou de ir até a orla do Guaíba, e ao longo do dia vão passar por lá **52 pessoas**, entre quem chega mais cedo e quem chega depois do almoço.

Você ficou responsável por comprar as bebidas. Pra te ajudar a carregar as caixas, tem **mais 5 amigos** — juntando com você, um grupinho de **6 pessoas**.

Antes de sair correndo pro quiosque, vem a pergunta que muda tudo:

> **Comprar bebida pra 52 pessoas baseado no gosto de quantas pessoas?**

Isso nos apresenta os dois primeiros conceitos:

- **População**: o grupo todo que queremos entender → as 52 pessoas do passeio.
- **Amostra**: uma parte da população, usada pra representar o todo → os 6 amigos do grupinho de compras.


In [ ]:
populacao_total = 52
amigos_no_grupo_de_compras = 6

print(f"População (todo mundo no passeio): {populacao_total} pessoas")
print(f"Amostra (grupinho que vai comprar as bebidas): {amigos_no_grupo_de_compras} pessoas")
print(f"\nA amostra representa {amigos_no_grupo_de_compras / populacao_total:.1%} da população.")


**Pense um pouco:** se você comprasse as bebidas baseado só no que **você** quer (uma "amostra" de 1 pessoa), qual seria o risco? E se perguntasse pras 6? E se conseguisse perguntar pras 52? Vamos comprovar essa intuição com números reais mais à frente.


***
## Capítulo 2 — Organizando o que a gente já sabe

Toda análise de dados começa organizando a informação numa **tabela**. Cada **coluna** (também chamada de **série** ou **variável**) representa uma categoria de informação, e cada **linha** (ou **observação**) representa as respostas de uma pessoa.

Você perguntou pros seus 5 amigos (e anotou a sua escolha também):


In [ ]:
grupo_compras = pd.DataFrame({
    "pessoa": ["Você", "Bibiana", "Rafa", "Duda", "Kelvin", "Vitória"],
    "bebida_escolhida": ["Guaraná", "Água de coco", "Guaraná", "Suco de laranja", "Guaraná", "Água"]
})

grupo_compras


Repare que essa tabela já pode ser **ordenada** (por exemplo, em ordem alfabética de bebida) ou **agrupada** (juntando quem escolheu a mesma coisa). Vamos ver as duas formas:


In [ ]:
# Ordenando por bebida (ordem alfabética)
print("📋 Ordenado por bebida:")
display(grupo_compras.sort_values("bebida_escolhida"))


In [ ]:
# Agrupando por bebida (juntando quem escolheu a mesma coisa)
print("📋 Agrupado por bebida:")
for bebida, grupo in grupo_compras.groupby("bebida_escolhida"):
    print(f"\n{bebida}:")
    print(grupo["pessoa"].to_list())


***
## Capítulo 3 — Quantas vezes cada bebida apareceu? (Frequência)

Agora vamos contar quantas vezes cada bebida foi escolhida. Isso é a **frequência**:

- **Frequência absoluta**: quantidade "crua" de vezes que aparece.
- **Frequência relativa**: a frequência absoluta dividida pelo total — normalmente em porcentagem.
- **Moda**: o valor de maior frequência (o "mais escolhido").


In [ ]:
frequencia_absoluta = grupo_compras["bebida_escolhida"].value_counts()
frequencia_relativa = grupo_compras["bebida_escolhida"].value_counts(normalize=True) * 100

tabela_frequencia = pd.DataFrame({
    "Frequência Absoluta": frequencia_absoluta,
    "Frequência Relativa (%)": frequencia_relativa.round(1)
})

tabela_frequencia


In [ ]:
moda = grupo_compras["bebida_escolhida"].mode()[0]
print(f"🏆 A moda (bebida mais escolhida) é: {moda}")


Agora, se usarmos essa frequência relativa pra estimar quanto comprar pras **52 pessoas**:


In [ ]:
estimativa_compra = (frequencia_relativa / 100 * populacao_total).round().astype(int)
print("🛒 Estimativa de quantas bebidas de cada tipo comprar para 52 pessoas:\n")
print(estimativa_compra)
print(f"\nTotal estimado: {estimativa_compra.sum()} bebidas")


Parece razoável... mas repare: 26 + 9 + 9 + 9 = **53**, uma pessoa a mais que as 52 do passeio! Isso não é erro de conta — é o efeito de arredondar cada categoria separadamente (0,667 arredonda pra cima em mais de uma categoria ao mesmo tempo). É um efeito bem comum em estatística: ao arredondar partes de um todo uma por uma, a soma pode "escapar" um pouco do total. Na prática, dá pra ajustar a última categoria pra fechar exatamente 52, se for importante bater certinho.

E o mais importante: será que 6 pessoas são suficientes pra confiar nesse número? Vamos descobrir no próximo capítulo.


***
## Capítulo 4 — Chega mais um, e tudo muda

Bem na hora de sair pro quiosque, chega o Renan, que topa ajudar a carregar. Ele quer **suco de laranja**. Agora o grupo tem **7 pessoas**.


In [ ]:
grupo_compras_v2 = pd.concat([
    grupo_compras,
    pd.DataFrame({"pessoa": ["Renan"], "bebida_escolhida": ["Suco de laranja"]})
], ignore_index=True)

freq_abs_v2 = grupo_compras_v2["bebida_escolhida"].value_counts()
freq_rel_v2 = grupo_compras_v2["bebida_escolhida"].value_counts(normalize=True) * 100

comparacao = pd.DataFrame({
    "Freq. Relativa com 6 pessoas (%)": frequencia_relativa.round(1),
    "Freq. Relativa com 7 pessoas (%)": freq_rel_v2.round(1)
}).fillna(0)

comparacao


Repare: a fatia do **suco de laranja** quase dobrou (de ~17% pra ~29%) só porque **uma pessoa a mais** entrou na conta!

> Quanto **menor** a amostra, mais uma única resposta pesa no resultado — e mais fácil a estimativa "balançar" e ficar diferente da realidade.

Vamos comprovar isso de verdade com uma simulação, usando um arquivo com respostas reais de 43 pessoas (`dados/cafe.csv`) como se fosse a nossa "população verdadeira" de referência.


In [ ]:
# Carregando os dados reais de uma pesquisa de café com 43 pessoas
cafe = pd.read_csv("dados/cafe.csv")
cafe.head(10)


In [ ]:
# Frequência relativa "real", usando as 43 respostas (nossa população de referência)
freq_real = cafe["tipo_bebida"].value_counts(normalize=True) * 100
print("☕ Frequência relativa REAL (população de referência, 43 pessoas):\n")
print(freq_real.round(1))


In [ ]:
# Agora vamos simular: e se a gente tivesse perguntado só pra uma AMOSTRA pequena?
# Vamos sortear amostras de tamanhos diferentes e comparar com o valor real.

random.seed(42)  # fixa o sorteio, pra todo mundo ver o mesmo resultado ao rodar

tamanhos_de_amostra = [5, 10, 20, 43]
bebida_alvo = "Café sem açúcar"  # vamos acompanhar essa bebida como exemplo

print(f"Frequência relativa REAL de '{bebida_alvo}': {freq_real[bebida_alvo]:.1f}%\n")

for n in tamanhos_de_amostra:
    amostra = cafe.sample(n=n, random_state=42)
    freq_amostra = (amostra["tipo_bebida"] == bebida_alvo).mean() * 100
    diferenca = abs(freq_amostra - freq_real[bebida_alvo])
    print(f"Amostra de {n:>2} pessoas -> estimativa: {freq_amostra:5.1f}%  "
          f"(diferença de {diferenca:4.1f} pontos percentuais em relação ao valor real)")


📌 **O que aconteceu:** conforme a amostra cresce (se aproximando do tamanho da população inteira), a estimativa se aproxima do valor real. Com amostras pequenas, o "erro" pode ser bem maior.

👉 **Experimente:** troque o `random_state=42` por outro número, ou troque `bebida_alvo` por outra bebida, e rode de novo. Você vai ver que com amostras pequenas o resultado muda bastante a cada sorteio — com amostras grandes, ele se mantém bem mais estável.


***
## Capítulo 5 — O quiosque não tem tudo

Vocês chegam animados no quiosque... e ele só vende **café** (expresso, com açúcar, sem açúcar, com leite, cappuccino, mocaccino). Nada de guaraná, suco ou água de coco!

Isso é super comum em análise de dados real: **a coleta é limitada pelas opções disponíveis**. Alguém também pode responder **"Não gosto"** — o que é uma resposta válida (significa "não compre nada de café pra essa pessoa").

Vamos olhar de novo pra pesquisa completa de 43 pessoas (`cafe.csv`), agora com todas as categorias, incluindo quem não gosta de café:


In [ ]:
tabela_frequencia_cafe = pd.DataFrame({
    "Frequência Absoluta": cafe["tipo_bebida"].value_counts(),
    "Frequência Relativa (%)": (cafe["tipo_bebida"].value_counts(normalize=True) * 100).round(1)
})

# Frequência acumulada (soma progressiva, ordenada da maior pra menor frequência)
tabela_frequencia_cafe["Frequência Acumulada"] = tabela_frequencia_cafe["Frequência Absoluta"].cumsum()

tabela_frequencia_cafe


In [ ]:
quem_nao_gosta = (cafe["tipo_bebida"] == "Não gosto").sum()
total_pessoas = len(cafe)
print(f"De {total_pessoas} pessoas, {quem_nao_gosta} não gostam de café "
      f"({quem_nao_gosta/total_pessoas:.1%}).")
print("Ou seja: pra essas pessoas, melhor levar outra coisa (água, suco) além do café!")


Agora vamos usar essa frequência real pra estimar a compra de café pras **52 pessoas do passeio** — já descontando quem não gosta de café:


In [ ]:
cafe_sem_os_que_nao_gostam = cafe[cafe["tipo_bebida"] != "Não gosto"]
freq_rel_cafe = cafe_sem_os_que_nao_gostam["tipo_bebida"].value_counts(normalize=True)

pessoas_que_tomam_cafe = round(populacao_total * (1 - quem_nao_gosta/total_pessoas))
estimativa_cafe = (freq_rel_cafe * pessoas_que_tomam_cafe).round().astype(int)

print(f"Das 52 pessoas, aproximadamente {pessoas_que_tomam_cafe} tomam café.")
print("\n🛒 Sugestão de compra de café por tipo:\n")
print(estimativa_cafe)


***
## Capítulo 6 — Quem é mais alto, quem é mais baixo, e qual o "meio-termo"?

Enquanto a fila anda, o assunto vira as alturas do grupinho de amigos. Vamos usar esse gancho pra aprender as **medidas de posição**:

- **Mínimo** / **Máximo**: menor e maior valor.
- **Média**: soma de tudo dividido pela quantidade de valores.
- **Mediana**: o valor "do meio" depois de ordenar os dados.


In [ ]:
alturas = pd.Series({
    "Você": 1.65,
    "Bibiana": 1.58,
    "Rafa": 1.70,
    "Duda": 1.61,
    "Kelvin": 1.95,
    "Vitória": 1.60,
})

print(f"Mínimo:  {alturas.min():.2f} m  ({alturas.idxmin()})")
print(f"Máximo:  {alturas.max():.2f} m  ({alturas.idxmax()})")
print(f"Média:   {alturas.mean():.3f} m")
print(f"Mediana: {alturas.median():.3f} m")


Repare que a **média** (1,68 m) ficou puxada pra cima por causa do Kelvin, que é bem mais alto que o resto do grupo (1,95 m). Já a **mediana** (1,63 m) representa melhor "a altura típica" do grupinho, porque não é tão sensível a um valor fora do padrão.

> 💡 Regra prática: quando existem valores muito "fora da curva" nos seus dados, prefira olhar também a mediana, não só a média.


***
## Capítulo 7 — Vendo os dados em vez de só ler números

Números em tabela são úteis, mas nosso cérebro entende **padrões visuais** muito mais rápido. Vamos representar a pesquisa de café com diferentes tipos de gráfico.


In [ ]:
contagem_cafe = cafe["tipo_bebida"].value_counts()

# Gráfico de barras horizontais
fig, ax = plt.subplots()
contagem_cafe.sort_values().plot(kind="barh", ax=ax, color="#6f4e37")
ax.set_xlabel("Frequência absoluta")
ax.set_title("☕ Preferência de café — Gráfico de barras")
plt.tight_layout()
plt.show()


In [ ]:
# Gráfico de pizza (frequência relativa)
fig, ax = plt.subplots()
contagem_cafe.plot(kind="pie", autopct="%1.0f%%", ax=ax, ylabel="")
ax.set_title("☕ Preferência de café — Gráfico de pizza")
plt.tight_layout()
plt.show()


In [ ]:
# Gráfico de pontos (dotplot) - uma alternativa simples e visual à frequência
fig, ax = plt.subplots()
for i, (bebida, freq) in enumerate(contagem_cafe.items()):
    ax.scatter([i] * freq, range(1, freq + 1), color="#6f4e37", s=80)
ax.set_xticks(range(len(contagem_cafe)))
ax.set_xticklabels(contagem_cafe.index, rotation=45, ha="right")
ax.set_ylabel("Quantidade de pessoas (empilhado)")
ax.set_title("☕ Preferência de café — Gráfico de pontos (dotplot)")
plt.tight_layout()
plt.show()


👉 **Pra pensar:** qual desses três gráficos te ajudou mais rápido a perceber qual é o café mais pedido? Isso não tem resposta única — depende do que você quer destacar. O de pizza é ótimo pra mostrar "fatia do todo"; o de barras é ótimo pra comparar categorias rapidamente.


***
## Capítulo 8 — O quanto os dados "variam" (Desvio Padrão)

Voltando às alturas: dois grupos podem ter a mesma média, só que um é mais "parecido" entre si e o outro tem gente bem diferente. A média sozinha não conta isso — pra isso existe o **desvio padrão**.

Ele mede o quanto os valores costumam se afastar da média, em média:

1. **Variância** = média do quadrado da diferença entre cada valor e a média.
2. **Desvio padrão** = raiz quadrada da variância.


In [ ]:
media_alturas = alturas.mean()
diferencas = alturas - media_alturas
diferencas_ao_quadrado = diferencas ** 2

variancia = diferencas_ao_quadrado.mean()
desvio_padrao = np.sqrt(variancia)

tabela_desvio = pd.DataFrame({
    "Altura": alturas,
    "Diferença p/ média": diferencas.round(3),
    "Diferença ao quadrado": diferencas_ao_quadrado.round(4),
})
display(tabela_desvio)

print(f"\nMédia: {media_alturas:.3f} m")
print(f"Variância: {variancia:.4f}")
print(f"Desvio padrão: {desvio_padrao:.3f} m")
print("\n(Confirmando com a função pronta do pandas:", round(alturas.std(ddof=0), 3), "m)")


Um desvio padrão de aproximadamente 0,12 m significa que, em geral, as alturas do grupo variam cerca de 12 centímetros pra cima ou pra baixo da média. Se todo mundo tivesse praticamente a mesma altura, o desvio padrão seria bem pertinho de zero.

💡 Esse mesmo raciocínio é usado, por exemplo, por bancos e aplicativos de cartão de crédito: se uma compra está muito "fora do desvio padrão" dos seus gastos normais, ela pode ser sinalizada como possível fraude.


***
## Capítulo 9 — Existe relação entre duas coisas? (Correlação)

Alguém do grupo brinca: "será que tem a ver a idade da pessoa com o tipo de café que ela pede?" Isso é uma pergunta sobre **correlação**.

- Correlação **próxima de 1**: quando uma variável sobe, a outra tende a subir também.
- Correlação **próxima de -1**: quando uma sobe, a outra tende a descer.
- Correlação **próxima de 0**: não existe relação linear perceptível.

Vamos testar com um exemplo numérico clássico: altura e idade de um grupo de jogadores.


In [ ]:
jogadores = pd.DataFrame({
    "idade": [15, 16, 17, 19, 21],
    "altura": [1.58, 1.60, 1.70, 1.75, 1.80],
})

correlacao = jogadores["idade"].corr(jogadores["altura"])
print(f"Correlação entre idade e altura: {correlacao:.3f}")

fig, ax = plt.subplots()
ax.scatter(jogadores["idade"], jogadores["altura"], s=100, color="#2a6f97")
ax.set_xlabel("Idade")
ax.set_ylabel("Altura (m)")
ax.set_title("Idade x Altura")
plt.tight_layout()
plt.show()


A correlação ficou bem próxima de 1: nesse grupo, quanto maior a idade, maior a altura (o que faz sentido, já que são adolescentes ainda crescendo).

⚠️ **Cuidado importante:** correlação **não é** a mesma coisa que causa e efeito. Existem correlações "espúrias" — números que batem sem nenhuma relação lógica real (tipo uma correlação famosa entre a taxa de divórcio em um estado americano e o consumo de margarina por lá!). Você pode se divertir com mais exemplos assim em [tylervigen.com/spurious-correlations](http://www.tylervigen.com/spurious-correlations).


***
## Capítulo 10 — Prevendo valores que a gente não tem (Regressão, Interpolação e Extrapolação)

Quando duas variáveis têm correlação forte, dá pra construir uma **regressão linear**: uma reta que descreve essa relação e permite **estimar** valores que você não mediu diretamente.

- **Interpolação**: estimar um valor **dentro** da faixa que você já tem dados.
- **Extrapolação**: estimar um valor **fora** dessa faixa — mais arriscado, porque assume que o padrão continua igual além do que foi observado.


In [ ]:
# Ajustando uma reta (regressão linear) para idade x altura
coeficientes = np.polyfit(jogadores["idade"], jogadores["altura"], deg=1)
inclinacao, intercepto = coeficientes

def prever_altura(idade):
    return inclinacao * idade + intercepto

print(f"Equação da reta: altura = {inclinacao:.4f} * idade + {intercepto:.4f}\n")

# Interpolação: idade 18, que está DENTRO do intervalo observado (15 a 21)
print(f"Interpolação -> altura estimada aos 18 anos: {prever_altura(18):.2f} m")

# Extrapolação: idade 30, que está FORA do intervalo observado
print(f"Extrapolação -> altura 'estimada' aos 30 anos: {prever_altura(30):.2f} m  ⚠️ pouco confiável!")


In [ ]:
fig, ax = plt.subplots()
ax.scatter(jogadores["idade"], jogadores["altura"], s=100, color="#2a6f97", label="Dados reais")

idades_para_reta = np.linspace(10, 32, 50)
ax.plot(idades_para_reta, prever_altura(idades_para_reta), color="#e63946", linestyle="--", label="Reta de regressão")

ax.scatter([18], [prever_altura(18)], s=140, color="green", zorder=5, label="Interpolação (18 anos)")
ax.scatter([30], [prever_altura(30)], s=140, color="orange", zorder=5, label="Extrapolação (30 anos)")

ax.set_xlabel("Idade")
ax.set_ylabel("Altura (m)")
ax.set_title("Regressão linear: idade x altura")
ax.legend()
plt.tight_layout()
plt.show()


Veja como a extrapolação pra 30 anos gera um valor pouco realista (mais de 2 metros!). Isso acontece porque uma pessoa não cresce pra sempre no mesmo ritmo — o padrão observado em adolescentes não se mantém em adultos. **Extrapolar exige cuidado redobrado.**


***
## Capítulo 11 — Estudo de caso real: quanto custa ir até o Guaíba de carro?

Depois do café, a conversa vai pro carro que trouxe o grupo. A família registrou, ao longo de vários anos, **cada abastecimento**: data, distância percorrida, preço do álcool e da gasolina, valor pago e litros abastecidos. Isso está no arquivo `dados/combustivel.csv`, com **65 registros**.

Vamos aplicar a **metodologia de análise de dados em 6 passos**:

1. Definir perguntas
2. Definir o que medir
3. Definir como medir
4. Coletar os dados
5. Analisar os dados
6. Interpretar os resultados

**Pergunta que queremos responder:** *"Em média, quanto se gasta de combustível por quilômetro rodado, e dá pra estimar o custo de uma viagem futura com base nisso?"*


In [ ]:
combustivel = pd.read_csv("dados/combustivel.csv", sep=";", decimal=",")
combustivel["Data"] = pd.to_datetime(combustivel["Data"], format="%d/%m/%Y")
combustivel["Km_por_litro"] = combustivel["Percorrida"] / combustivel["Litros"]

combustivel.head(10)


### Passo 5 — Análise descritiva


In [ ]:
print("📊 Estatísticas descritivas gerais:\n")
print(f"Distância total percorrida: {combustivel['Percorrida'].sum():,.1f} km")
print(f"Total de combustível: {combustivel['Litros'].sum():,.1f} litros")
print(f"Valor total gasto: R$ {combustivel['Valor'].sum():,.2f}")
print(f"Rendimento médio: {combustivel['Km_por_litro'].mean():.2f} km/l")
print(f"Rendimento mínimo: {combustivel['Km_por_litro'].min():.2f} km/l")
print(f"Rendimento máximo: {combustivel['Km_por_litro'].max():.2f} km/l")
print(f"Desvio padrão do rendimento: {combustivel['Km_por_litro'].std():.2f} km/l")


📌 **Reparou como o rendimento mínimo (perto de 1,5 km/l) e o máximo (perto de 45 km/l) estão bem longe da média (~12 km/l)?** Nenhum carro comum faz 1,5 km/l de verdade! Isso é um sinal de **dado "sujo"**: provavelmente algumas vezes o tanque foi completado só parcialmente, então a conta de "distância desse abastecimento ÷ litros desse abastecimento" não representa o rendimento real do carro naquele trecho.

Isso é muito comum em dados do mundo real — e é exatamente por isso que o passo 5 da metodologia (**analisar os dados**) muitas vezes revela que é preciso voltar no passo 4 (**coleta**) pra entender ou limpar esses casos, em vez de confiar cegamente em qualquer número calculado. Por enquanto, vamos seguir com os dados como estão, mas de olho nesse detalhe.


### Passo 5 (continuação) — Análise exploratória: existe correlação entre distância e litros consumidos?

A intuição diz que sim: quanto mais se roda, mais combustível se gasta. Vamos conferir com números — sem supor a resposta antes de calcular.


In [ ]:
correlacao_km_litros = combustivel["Percorrida"].corr(combustivel["Litros"])
print(f"Correlação entre distância percorrida e litros consumidos: {correlacao_km_litros:.3f}")

fig, ax = plt.subplots()
ax.scatter(combustivel["Percorrida"], combustivel["Litros"], color="#e76f51")
ax.set_xlabel("Distância percorrida (km)")
ax.set_ylabel("Litros consumidos")
ax.set_title("Distância x Consumo de combustível")
plt.tight_layout()
plt.show()


O valor encontrado (por volta de 0,37) é **positivo, mas moderado** — bem mais fraco do que a intuição sugeria. Olhando o gráfico de dispersão dá pra entender por quê: existem vários abastecimentos com distância parecida mas quantidade de litros bem diferente, exatamente os tais abastecimentos parciais que identificamos acima.

Essa é uma lição valiosa: **a intuição nem sempre bate com o que os dados mostram**, e às vezes isso acontece porque a coleta de dados tem ruído, não porque a relação entre as variáveis realmente não existe. Um próximo passo de uma análise mais cuidadosa seria filtrar apenas abastecimentos completos antes de calcular a correlação — fica de exercício pra quem quiser se aprofundar!

Agora vamos ver a evolução do custo ao longo do tempo, com um gráfico de linha:


In [ ]:
fig, ax = plt.subplots()
ax.plot(combustivel["Data"], combustivel["Valor"], marker="o", markersize=3, color="#264653")
ax.set_xlabel("Data")
ax.set_ylabel("Valor pago (R$)")
ax.set_title("Evolução do valor pago por abastecimento ao longo do tempo")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


### Passo 6 — Interpretando e estimando (regressão + extrapolação)

Vamos construir uma regressão linear simples de **litros consumidos em função da distância percorrida**, e usá-la pra estimar quanto combustível seria necessário para uma viagem de uma distância específica.


In [ ]:
coef = np.polyfit(combustivel["Percorrida"], combustivel["Litros"], deg=1)
inclinacao_comb, intercepto_comb = coef

def estimar_litros(distancia_km):
    return inclinacao_comb * distancia_km + intercepto_comb

# Exemplo: uma viagem de 300 km (dentro da faixa observada -> interpolação)
distancia_exemplo = 300
litros_estimados = estimar_litros(distancia_exemplo)
preco_medio_gasolina = combustivel["Gasolina"].mean()
custo_estimado = litros_estimados * preco_medio_gasolina

print(f"Para uma viagem de {distancia_exemplo} km:")
print(f"  Litros estimados: {litros_estimados:.1f} L")
print(f"  Preço médio da gasolina no histórico: R$ {preco_medio_gasolina:.2f}")
print(f"  Custo estimado: R$ {custo_estimado:.2f}")


👉 **Experimente:** troque `distancia_exemplo` pela distância de uma viagem que você conhece (por exemplo, a distância da sua cidade até Porto Alegre) e veja a estimativa de custo. Lembre-se: se a distância for **muito maior** do que qualquer valor já visto no histórico, você estaria extrapolando — e o resultado deve ser visto com mais desconfiança, já que o preço do combustível muda com o tempo e o carro pode se comportar diferente em estradas muito mais longas.

### Recapitulando a metodologia aplicada:

| Passo | O que fizemos |
|---|---|
| 1. Definir perguntas | Quanto custa rodar uma certa distância? |
| 2. Definir o que medir | Distância, litros, preço, valor pago |
| 3. Definir como medir | Usar o histórico de abastecimentos já registrado |
| 4. Coletar dados | `combustivel.csv`, 65 registros |
| 5. Analisar dados | Médias, correlação, gráficos, regressão |
| 6. Interpretar | Estimativa de custo, com ressalva sobre extrapolação |


***
## Capítulo 12 — Análise qualitativa x quantitativa (recapitulando)

Ao longo da tarde, usamos sempre a **análise quantitativa**: números, frequências, médias, correlações. Ela responde "o quê" e "quanto".

Se, em vez disso, alguém perguntasse "o que vocês acharam do passeio?" e as respostas fossem frases livres tipo *"foi tri bom, mas a fila do quiosque demorou"*, precisaríamos fazer uma **análise qualitativa**: ler as respostas, identificar palavras-chave e temas em comum (ex: "elogios ao clima", "reclamação sobre demora"), e só depois, se quiséssemos, transformar isso em números (quantas pessoas mencionaram "demora", por exemplo).

Os dois tipos se completam: números mostram **o quê**; opiniões ajudam a entender **o porquê**.


***
## Capítulo 13 — Seu desafio: aplique isso na sua vida 🎯

Agora é sua vez! Escolha um tema simples do seu dia a dia, por exemplo:

- Quanto você (ou sua família) gasta por mês em transporte, lanche, dados de internet.
- Quanto tempo você passa por dia em algum aplicativo, assistindo TV, ou no ônibus.
- Quantas vezes por semana você consome algo específico (chimarrão, café, um lanche).

**Passos sugeridos:**
1. Anote os dados por pelo menos 2-3 semanas.
2. Organize numa tabela (pode usar o `pandas`, como fizemos aqui).
3. Calcule frequência, média, mínimo, máximo e desvio padrão.
4. Construa pelo menos dois gráficos diferentes.
5. Escreva de 3 a 5 frases interpretando o que os números te mostraram.

Abaixo tem um "esqueleto" de código pronto — é só substituir os dados de exemplo pelos seus:


In [ ]:
# 🔧 ESQUELETO PARA VOCÊ PREENCHER COM SEUS PRÓPRIOS DADOS
# Troque os valores abaixo pelos seus dados reais coletados ao longo de 2-3 semanas.

meus_dados = pd.DataFrame({
    "data": pd.date_range("2026-09-01", periods=14, freq="D"),
    "valor": [22, 22, 22, 50, 0, 22, 22, 22, 22, 22, 0, 50, 22, 22],  # exemplo: gasto diário em R$
})

print("📋 Meus dados:")
display(meus_dados)

print(f"\nMédia: {meus_dados['valor'].mean():.2f}")
print(f"Mediana: {meus_dados['valor'].median():.2f}")
print(f"Mínimo: {meus_dados['valor'].min():.2f}")
print(f"Máximo: {meus_dados['valor'].max():.2f}")
print(f"Desvio padrão: {meus_dados['valor'].std():.2f}")

fig, ax = plt.subplots()
ax.plot(meus_dados["data"], meus_dados["valor"], marker="o", color="#588157")
ax.set_title("Meus dados ao longo do tempo")
ax.set_ylabel("Valor")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

fig, ax = plt.subplots()
ax.hist(meus_dados["valor"], bins=5, color="#a3b18a", edgecolor="black")
ax.set_title("Histograma dos meus dados")
ax.set_xlabel("Faixa de valor")
ax.set_ylabel("Frequência")
plt.tight_layout()
plt.show()


**Perguntas pra responder por escrito, depois de rodar seu próprio caso:**

1. Existe algum valor bem fora do padrão (muito diferente da média)? O que pode ter causado isso?
2. A média e a mediana são parecidas ou bem diferentes? O que isso te diz?
3. O desvio padrão é "alto" ou "baixo" em relação à média? O que isso significa sobre a sua rotina?
4. Se você tivesse coletado só os 3 primeiros dias como "amostra", a estimativa teria sido parecida com o resultado final?

***

## 🎓 Conceitos que você usou nesse notebook

População • Amostra • Variável / Série • Observação • Frequência absoluta • Frequência relativa • Frequência acumulada • Moda • Média • Mediana • Mínimo/Máximo • Desvio padrão • Correlação • Correlação espúria • Regressão linear • Interpolação • Extrapolação • Análise quantitativa/qualitativa • Metodologia de análise de dados (6 passos) • Gráficos de barras, pizza, pontos, linha e histograma

Parabéns por chegar até aqui! 🎉 Agora é só continuar praticando com dados do seu próprio dia a dia.
